# SHAP Interpretability Analysis

## Scientific objective

Quantify feature contributions to the XGBoost model predictions using SHAP (SHapley Additive exPlanations). This notebook computes SHAP values for the held-out test set and generates publication-quality interpretability figures.

## Background

SHAP provides consistent, model-agnostic explanations by estimating each feature's contribution to individual predictions based on Shapley values from cooperative game theory.

## Methodology

- Load the finalized feature matrix and target values.
- Retrain or load a fitted XGBoost regressor for interpretation.
- Compute SHAP values using a TreeExplainer.
- Generate summary, bar, dependence, and descriptor-only plots and save them as PNG and PDF at 300 dpi.

In [ ]:
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
from pathlib import Path

# Use relative, portable paths
ROOT = Path("..")
X = pd.read_csv(ROOT / "data" / "processed" / "X_combined.csv")
y = pd.read_csv(ROOT / "data" / "processed" / "y_pIC50.csv").squeeze()

print(X.shape)
print(y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## Training the Best Performing Model

The XGBoost model was retrained to enable SHAP-based interpretation of molecular descriptors and fingerprint features contributing to anti-leishmanial activity predictions.

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=1
)

xgb.fit(
    X_train,
    y_train
)

## SHAP Explainability Analysis

SHAP (SHapley Additive exPlanations) values were calculated to quantify the contribution of each feature to model predictions and identify molecular properties associated with anti-leishmanial activity.

In [ ]:
explainer = shap.TreeExplainer(xgb)

# Obtain SHAP values for the test set. For regressors this returns an array
shap_values = explainer.shap_values(X_test)

print(np.array(shap_values).shape)

In [ ]:
# Create and save the SHAP summary plot (be explicit about the figure)
shap.summary_plot(shap_values, X_test, show=False)
fig = plt.gcf()
fig.savefig(Path('..') / 'figures' / 'shap_summary_plot.png', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(Path('..') / 'figures' / 'shap_summary_plot.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig)

In [ ]:
# Bar-type SHAP importance plot
shap.summary_plot(shap_values, X_test, plot_type='bar', show=False)
fig = plt.gcf()
fig.savefig(Path('..') / 'figures' / 'shap_summary_barplot.png', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(Path('..') / 'figures' / 'shap_summary_barplot.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig)

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_test.columns,
    "Mean_Abs_SHAP": np.abs(shap_values).mean(axis=0)
})

feature_importance = feature_importance.sort_values(
    "Mean_Abs_SHAP",
    ascending=False
)

feature_importance.head(20)

In [ ]:
feature_importance.to_csv(
    "../results/shap_feature_importance.csv",
    index=False
)

In [ ]:
descriptor_names = [
    "Molecular_Weight",
    "AlogP",
    "TPSA",
    "HBA",
    "HBD",
    "RO5_Violations",
    "Rotatable_Bonds",
    "QED",
    "Aromatic_Rings",
    "Heavy_Atoms",
    "NP_Likeness"
]

feature_importance[
    feature_importance["Feature"].isin(
        descriptor_names
    )
]

In [ ]:
# Dependence plot for AlogP
shap.dependence_plot('AlogP', shap_values, X_test, show=False)
fig = plt.gcf()
fig.savefig(Path('..') / 'figures' / 'shap_dependence_alogp.png', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(Path('..') / 'figures' / 'shap_dependence_alogp.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig)

In [ ]:
# Dependence plot for Molecular Weight
shap.dependence_plot('Molecular_Weight', shap_values, X_test, show=False)
fig = plt.gcf()
fig.savefig(Path('..') / 'figures' / 'shap_dependence_mw.png', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(Path('..') / 'figures' / 'shap_dependence_mw.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig)

In [ ]:
descriptor_names = [
    'Molecular_Weight',
    'AlogP',
    'TPSA',
    'HBA',
    'HBD',
    'RO5_Violations',
    'Rotatable_Bonds',
    'QED',
    'Aromatic_Rings',
    'Heavy_Atoms',
    'NP_Likeness'
]

# Ensure all descriptor names are present
present_desc = [c for c in descriptor_names if c in X_test.columns]
desc_idx = [X_test.columns.get_loc(col) for col in present_desc]

# Descriptor-only SHAP values and summary plot
shap_desc = shap_values[:, desc_idx]
X_test_desc = X_test[present_desc]
shap.summary_plot(shap_desc, X_test_desc, show=False)
fig = plt.gcf()
fig.savefig(Path('..') / 'figures' / 'shap_descriptors_summary.png', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(Path('..') / 'figures' / 'shap_descriptors_summary.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig)

In [ ]:
descriptor_shap = pd.DataFrame({
    "Feature": descriptor_names,
    "Mean_Abs_SHAP": np.abs(shap_desc).mean(axis=0)
}).sort_values("Mean_Abs_SHAP", ascending=False)

descriptor_shap

## SHAP Interpretation of Descriptor Features

SHAP analysis showed that AlogP was the most influential molecular descriptor in the model, indicating that lipophilicity played the strongest role in anti-leishmanial activity prediction. Heavy atom count, TPSA, QED, HBD, and NP-Likeness also contributed to model predictions, while RO5 violations and HBA had comparatively low importance. The results suggest that activity is driven by a combination of physicochemical properties rather than a single descriptor alone.